# NCBI Entrez Tools Examples

This notebook demonstrates the three NCBI Entrez tools:

- **`run_ncbi_esearch`** -- Search for IDs by query term
- **`run_ncbi_esummary`** -- Retrieve record summaries by ID
- **`run_ncbi_efetch`** -- Fetch sequences in FASTA or other formats

## Imports

In [1]:
from proto_tools import (
    NCBIEfetchInput,
    NCBIEsearchInput,
    NCBIEsummaryInput,
    NCBIFetchConfig,
    run_ncbi_efetch,
    run_ncbi_esearch,
    run_ncbi_esummary,
)


## API Reference

### `NCBIEsearchInput`

| Field | Type | Default | Description |
|---|---|---|---|
| `db` | `Literal["protein", "nuccore", "gene"]` | *(required)* | NCBI database to query |
| `search_term` | `str` | *(required)* | NCBI search query |
| `max_results` | `int` | `5` | Maximum number of IDs to return |

### `NCBIEsummaryInput`

| Field | Type | Default | Description |
|---|---|---|---|
| `db` | `Literal["protein", "nuccore", "gene"]` | *(required)* | NCBI database to query |
| `identifier` | `str` | *(required)* | Accession or NCBI ID |

### `NCBIEfetchInput`

| Field | Type | Default | Description |
|---|---|---|---|
| `db` | `Literal["protein", "nuccore", "gene"]` | *(required)* | NCBI database to query |
| `identifier` | `str` | *(required)* | Accession or NCBI ID |
| `return_format` | `Literal["fasta", "fasta_cds_na"]` | `"fasta"` | NCBI rettype: `fasta` for sequences, `fasta_cds_na` for CDS |
| `seq_start` | `Optional[int]` | `None` | Start position (1-indexed, inclusive) |
| `seq_stop` | `Optional[int]` | `None` | Stop position (1-indexed, inclusive) |
| `strand` | `Optional[Literal["+", "-"]]` | `None` | Strand for nucleotide retrieval |

### `NCBIFetchConfig` (shared)

| Field | Type | Default | Description |
|---|---|---|---|
| `request_timeout_seconds` | `int` | `15` | HTTP timeout in seconds |
| `http_retries` | `int` | `2` | Retries for HTTP requests |
| `backoff_seconds` | `float` | `1.0` | Seconds to wait between retries (doubles after each attempt) |
| `ncbi_api_key` | `Optional[str]` | `None` | Optional NCBI API key |
| `ncbi_email` | `Optional[str]` | `None` | Optional NCBI email |
| `user_agent` | `str` | `"proto-tools/ncbi-fetch-v1"` | Identifier string sent to database APIs with each request |

### Output Types

**`NCBIEsearchOutput`** — `ids: List[str]`

**`NCBIEsummaryOutput`** — `summary: Dict[str, Any]`, `source_url: str`

**`NCBIEfetchOutput`** — `fasta_records: List[NCBIFastaRecord]`, `source_url: str`

## 1. Search for Protein IDs (esearch)

Search the NCBI protein database for lac repressor in E. coli.

In [2]:
inputs = NCBIEsearchInput(
    db="protein",
    search_term='"lacI"[Gene] AND "Escherichia coli"[Organism]',
    max_results=5,
)

output = run_ncbi_esearch(inputs, NCBIFetchConfig())

print(f"IDs found: {output.ids}")

IDs found: ['2731963152', '3181566493', '3055751481', '3174388526', '3172598205']


## 2. Fetch a Protein FASTA (efetch)

Fetch the TP53 protein sequence from NCBI by RefSeq accession.

In [3]:
inputs = NCBIEfetchInput(
    db="protein",
    identifier="NP_000537.3",
    return_format="fasta",
)

output = run_ncbi_efetch(inputs, NCBIFetchConfig())

record = output.fasta_records[0]
print(f"Accession: {record.accession}")
print(f"Header: {record.header[:80]}...")
print(f"Sequence length: {len(record.sequence)}")
print(f"Preview: {record.sequence[:60]}...")

Accession: NP_000537.3
Header: NP_000537.3 cellular tumor antigen p53 isoform a [Homo sapiens]...
Sequence length: 393
Preview: MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP...


## 3. Fetch a Genomic Subsequence

Retrieve a specific region of the E. coli K-12 genome using coordinates and strand.

In [4]:
# Fetch a 200bp region from the lacI locus (minus strand)
inputs = NCBIEfetchInput(
    db="nuccore",
    identifier="NC_000913.3",
    return_format="fasta",
    seq_start=366428,
    seq_stop=366628,
    strand="-",
)

output = run_ncbi_efetch(inputs, NCBIFetchConfig())

record = output.fasta_records[0]
print(f"Accession: {record.accession}")
print(f"Sequence length: {len(record.sequence)}")
print(f"Preview: {record.sequence[:80]}...")
print(f"Source URL: {output.source_url}")

Accession: NC_000913.3:c366628-366428
Sequence length: 201
Preview: CTGCTGGGGCAAACCAGCGTGGACCGCTTGCTGCAACTCTCTCAGGGCCAGGCGGTGAAGGGCAATCAGCTGTTGCCCGT...
Source URL: https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id=NC_000913.3&rettype=fasta&retmode=text&seq_start=366428&seq_stop=366628&strand=2&tool=proto_tools_ncbi_fetch


## 4. Retrieve a Gene Summary (esummary)

Get summary metadata for a gene record, including genomic coordinates.

In [5]:
# First, find the gene ID for TP53 in humans
search_output = run_ncbi_esearch(
    NCBIEsearchInput(
        db="gene",
        search_term='"TP53"[Gene Name] AND "Homo sapiens"[Organism]',
        max_results=1,
    ),
    NCBIFetchConfig(),
)
gene_id = search_output.ids[0]
print(f"Gene ID: {gene_id}")

# Now get the gene summary
summary_output = run_ncbi_esummary(
    NCBIEsummaryInput(
        db="gene",
        identifier=gene_id,
    ),
    NCBIFetchConfig(),
)

gene_data = summary_output.summary.get(gene_id, {})
print(f"Name: {gene_data.get('name')}")
print(f"Description: {gene_data.get('description')}")
print(f"Organism: {gene_data.get('organism', {}).get('scientificname')}")

genomic_info = gene_data.get("genomicinfo", [])
if genomic_info:
    region = genomic_info[0]
    print(f"Chromosome: {region.get('chraccver')}")
    print(f"Coordinates: {region.get('chrstart')}-{region.get('chrstop')}")

Gene ID: 7157
Name: TP53
Description: tumor protein p53
Organism: Homo sapiens
Chromosome: NC_000017.11
Coordinates: 7687489-7668420


## Export Results

Save fetched results to JSON format.

In [6]:
# Export efetch result to JSON
output.export("ncbi_efetch_results", export_path="./example_output", file_format="json")
print("Exported to ./example_output/ncbi_efetch_results.json")

# Export gene summary to JSON
summary_output.export("ncbi_gene_summary", export_path="./example_output", file_format="json")
print("Exported to ./example_output/ncbi_gene_summary.json")

Exported to ./example_output/ncbi_efetch_results.json
Exported to ./example_output/ncbi_gene_summary.json
